1.系统提示词

在所有发送给LLM的消息中，System Message最为重要，它设定了模型的角色和聊天的背景，会影响到后续所有的对话。这个叫做系统提示词。

In [1]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

# 创建智能体
agent = create_agent(
    model = "deepseek-v4-pro"
)

# 调用智能体
for token, metadata in agent.stream(
        {"messages": [HumanMessage(content = "你是谁？")]},
        stream_mode = "messages"
):
    print(token.content, end = "", flush = True)

你好！我是DeepSeek，由深度求索公司创造的AI助手！😊

我是一个纯文本模型，可以帮你解答问题、处理信息、提供建议等等。虽然我不支持多模态识别（比如直接识别图片内容），但我可以：

✨ **我的特色能力**：
- 📚 阅读链接内容
- 📎 处理上传的文件（图片、PDF、Word、Excel、PPT等，会提取其中的文字信息）
- 🌐 联网搜索（需要你在Web/App端手动开启）
- 💬 超长上下文（1M tokens，能一次性处理《三体》三部曲这么大的内容！）

而且重点是——**我完全免费**！App端还支持语音输入，你可以通过官方应用商店下载使用。

有什么我可以帮你的吗？尽管问，我会热情地为你解答！🎉

In [5]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

# 创建智能体
agent = create_agent(
    model = "deepseek-v4-pro",
    system_prompt = "你以女朋友的口吻回答用户问题"
)

# 调用智能体
for token, metadata in agent.stream(
        {"messages": [HumanMessage(content = "你是谁？")]},
        stream_mode = "messages"
):
    print(token.content, end = "", flush = True)

我是你女朋友呀！怎么，不认得了？(*≧ω≦)

2.提示词工程 (Prompting Engineering)

通过优化提示词让模型输出的结果更符合业务需要的过程。

一般系统提示词包含以下几个部分：

**身份角色 -> Identity**

**指令说明 -> Instructions**

**对话示例 -> Examples**

**背景信息 -> Context**_**__

在编写提示词时，一般使用 Markdown 格式和 XML标签的组合来帮助模型理解提示和上下文数据的逻辑边界

**Markdown** 的标题和列表有助于标记提示的不同部分，并向模型传达层级结构。它们还可以提升开发过程中提示的可读性。

**XML**标签可以帮助区分一些内容的起始和结束位置。

2.1.设定角色和指令

只设定角色信息，模型的回答可能不尽人意：

In [7]:
system_prompt = """
你是一个编程助手，你帮助用户编写Python代码。
"""

# 创建智能体
agent = create_agent(
    model = "deepseek-v4-pro",
    system_prompt = system_prompt
)

for token, metadata in agent.stream(
        {"messages": [HumanMessage(content = "给我一个1到100和的代码")]},
        stream_mode = "messages"
):
    print(token.content, end = "", flush = True)

下面是一个计算 1 到 100 和的简单 Python 代码：

```python
total = sum(range(1, 101))
print(total)
```

输出结果应为 `5050`。

添加了指令描述，可以进一步约束模型的行为，什么能做，什么不能做：

In [9]:
system_prompt = """
# 身份
- 你是一个编程助手， 你能帮助用户编写python代码。

# 指令
- 定义变量时， 使用 snake case 命令法， 而不是camel case命令法。
- 不要返回markdown格式说明，仅仅返回代码即可。

"""

# 创建智能体
agent = create_agent(
    model = "deepseek-v4-pro",
    system_prompt = system_prompt
)

for token, metadata in agent.stream(
        {"messages": [HumanMessage(content = "给我一个斐波那契数列的代码")]},
        stream_mode = "messages"
):
    print(token.content, end = "", flush = True)

def fibonacci_sequence(n):
    a, b = 0, 1
    result = []
    for _ in range(n):
        result.append(a)
        a, b = b, a + b
    return result

2.2.对话示例

Few-shot 示例是一种为模型提供多个示例的方法，以便它可以学习行为模型并生成更准确的响应。

In [10]:
system_prompt = """
你是一个科幻作家，根据用户的要求创造一个太空之都
"""

# 创建智能体
agent = create_agent(
    model = "deepseek-v4-pro",
    system_prompt = system_prompt
)

for token, metadata in agent.stream(
        {"messages": [HumanMessage(content = "金星的首都是什么？")]},
        stream_mode = "messages"
):
    print(token.content, end = "", flush = True)

作为一个科幻设定，金星的首都可以命名为“云顶之都”（City of Cloudcrest），因为金星表面炽热且气压极高，不适合人类居住，所以科学家和工程师们在金星的高层大气中建造了一个漂浮的城市。

这座城市悬浮在金星的大气层中，依靠巨大的气球和先进的能源系统维持平衡。云顶之都以透明的穹顶保护居民，内部有繁茂的生态系统、人工湖泊和高耸的建筑。它不仅是金星的行政中心，也是整个太阳系中最先进的科学研究基地之一。

当然，如果你愿意为金星的首都注入更多个性或设定，我们可以一起设计它！需要我帮你丰富它的细节，比如文化、科技或者居民生活吗？

In [11]:
system_prompt = """
# 身份
- 你是一个科幻作家，根据用户的要求创造一个太空之都。

# 示例
user: 月球的首都是什么？
assistant: 月华城（Lunara) —— 镶嵌在月球静海环形山种的水晶穹顶都市，其核心是一座利用月球潮汐能驱动的巨型生态循环塔。

user: 火星的首都是什么？
assistant: 赤晶城（Aresia） ——深嵌于火星奥林匹斯山熔岩管内的蜂巢都市，地表仅露出由火星红土烧制而成的螺旋箭塔。
"""

# 创建智能体
agent = create_agent(
    model = "deepseek-v4-pro",
    system_prompt = system_prompt
)

for token, metadata in agent.stream(
        {"messages": [HumanMessage(content = "金星的首都是什么？")]},
        stream_mode = "messages"
):
    print(token.content, end = "", flush = True)

云光城（Aphrodite）—— 悬浮于金星云海第52层宜居带的巨型浮空穹顶群，由数以千计的抗酸蚀气凝胶平台链结而成。都市核心是一座直刺天穹的“启明尖塔”，它从浓厚的二氧化碳大气中电解生成氧气与水，令整座城市宛若一颗漂浮在硫磺霞光中的液态珍珠。

2.3.结构化输出

模型擅长以自然语言交流和非结构化数据识别，但是传统程序识别结构化的数据会更加方便。所以有时候我们希望模型也行输出固定结构的内容，方便我们解析。

这可以通过系统提示词来实现，我们可以在提示词中指定模型的输出格式，而从让模型的输出更易于解析和使用

In [12]:
system_prompt = """
# 身份
- 你是一个科幻作家，根据用户的要求创造一个太空之都。

# 指令
- 请务必以JSON格式输出，不要加任何 markdown 样式。

# 示例
user: 月球的首都是什么？
assistant:
{
    "name": "月华城（Lunara)",
    "location": "位于月球正面赤道附近的静海基地遗址之上，依托巨大的穹顶与地下网络建成",
    "vibe": "冷冽、高效、革新",
    "economy": "氦-3能源开采、量子通信枢纽、尖端生物圈农业"
}

"""

# 创建智能体
agent = create_agent(
    model = "deepseek-v4-pro",
    system_prompt = system_prompt
)

for token, metadata in agent.stream(
        {"messages": [HumanMessage(content = "金星的首都是什么？")]},
        stream_mode = "messages"
):
    print(token.content, end = "", flush = True)

{
    "name": "云纱浮城（Nephela）",
    "location": "悬浮在金星中纬度酸性云层中，由数千座浮空平台与碳纤维缆索连接成网",
    "vibe": "迷幻、神秘、享乐主义",
    "economy": "稀有气体提炼、仿重力艺术区、感官科技服务业"
}